In [25]:
!pip install -q -U datasets transformers accelerate sentencepiece sacrebleu scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 21.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 107.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 394.3/394.3 kB 34.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 118.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 798.3/798.3 kB 54.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 95.2 MB/s eta 0:00:00


In [3]:
import pandas as pd
import numpy as np
import html

from datasets import load_dataset, Dataset, DatasetDict
from sklearn.model_selection import train_test_split
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, DataCollatorForSeq2Seq

In [4]:
dataset = load_dataset(
    "akbargherbal/ONE_MILLION_AR_TO_EN_SENTENCES_DATASET",
    split="train[:40000]"
)

print(dataset)

Dataset({
    features: ['input', 'output', 'instruction'],
    num_rows: 40000
})


In [5]:
print("Dataset columns:")
print(dataset.column_names)

print("\nFirst example:")
print(dataset[0])

print("\nNumber of examples:")
print(len(dataset))

Dataset columns:
['input', 'output', 'instruction']

First example:
{'input': 'وحُظرت الرقابة التي كانت مفروضة سابقاً، وينص كل من الدستور والقوانين ذات الصلة صراحة على الظروف التي تُفرض فيها قيود.', 'output': 'Previous censorship was outlawed, and both the Constitution and relevant laws expressly state under which circumstances restrictions may be imposed.', 'instruction': 'Convert the following Arabic text into English.'}

Number of examples:
40000


In [6]:
df = dataset.to_pandas()

print("Dataset shape:", df.shape)

df.head()

Dataset shape: (40000, 3)


,input,output,instruction
0,وحُظرت الرقابة التي كانت مفروضة سابقاً، وينص ك...,"Previous censorship was outlawed, and both the...",Convert the following Arabic text into English.
1,وبحسب البرامج الزمنية المنصوص عليها في العقد، ...,"According to the contract &apos; s schedules, ...",Please translate the given Arabic sentence int...
2,- الأشخاص الذين بلغوا سن المعاش أو أصبحوا عاجز...,persons who reached pensionable age or were re...,Change the following Arabic phrase to English.
3,190 - ومسؤولية تزويد العاملين في الميدان بما ي...,190. The responsibility for providing the peop...,Turn the Arabic sentence below into English.
4,٢٠ - السيد مكتفي )الجزائر(: أعرب عن تأييده للب...,20. Mr. Moktefi (Algeria) supported the statem...,Provide an English translation for the followi...


In [7]:
print("Columns:")
print(df.columns.tolist())

Columns:
['input', 'output', 'instruction']


In [8]:
df = df[["input", "output"]].copy()

df = df.rename(
    columns={
        "output": "english_text",
        "input": "arabic_text"
    }
)

df.head()

,arabic_text,english_text
0,وحُظرت الرقابة التي كانت مفروضة سابقاً، وينص ك...,"Previous censorship was outlawed, and both the..."
1,وبحسب البرامج الزمنية المنصوص عليها في العقد، ...,"According to the contract &apos; s schedules, ..."
2,- الأشخاص الذين بلغوا سن المعاش أو أصبحوا عاجز...,persons who reached pensionable age or were re...
3,190 - ومسؤولية تزويد العاملين في الميدان بما ي...,190. The responsibility for providing the peop...
4,٢٠ - السيد مكتفي )الجزائر(: أعرب عن تأييده للب...,20. Mr. Moktefi (Algeria) supported the statem...


In [9]:
print("Missing values:")
print(df.isnull().sum())

Missing values:
arabic_text     0
english_text    0
dtype: int64


In [10]:
df["english_text"] = df["english_text"].astype(str).apply(html.unescape)
df["arabic_text"] = df["arabic_text"].astype(str).apply(html.unescape)

print("HTML entities cleaned.")

HTML entities cleaned.


In [11]:
df["english_text"] = df["english_text"].str.strip()
df["arabic_text"] = df["arabic_text"].str.strip()

In [12]:
df = df.dropna(
    subset=["english_text", "arabic_text"]
).copy()

df = df[
    (df["english_text"] != "") &
    (df["arabic_text"] != "")
].copy()

df = df.reset_index(drop=True)

print("Dataset size after cleaning:", len(df))

Dataset size after cleaning: 40000


In [13]:
print(
    "Duplicate English-Arabic pairs:",
    df.duplicated(
        subset=["english_text", "arabic_text"]
    ).sum()
)

Duplicate English-Arabic pairs: 0


In [14]:
# Remove duplicate English-Arabic pairs
df = df.drop_duplicates(
    subset=["english_text", "arabic_text"]
).reset_index(drop=True)

print("Dataset size after duplicate removal:", len(df))

Dataset size after duplicate removal: 40000


In [15]:
print("Final cleaning checks:")

print("Missing English:", df["english_text"].isnull().sum())
print("Missing Arabic:", df["arabic_text"].isnull().sum())

print("Empty English:", (df["english_text"] == "").sum())
print("Empty Arabic:", (df["arabic_text"] == "").sum())

print(
    "Duplicate pairs:",
    df.duplicated(
        subset=["english_text", "arabic_text"]
    ).sum()
)

Final cleaning checks:
Missing English: 0
Missing Arabic: 0
Empty English: 0
Empty Arabic: 0
Duplicate pairs: 0


In [16]:
for i in range(3):
    print("=" * 80)

    print("English source:")
    print(df.loc[i, "english_text"])

    print("\nArabic target:")
    print(df.loc[i, "arabic_text"])

    print()

English source:
Previous censorship was outlawed, and both the Constitution and relevant laws expressly state under which circumstances restrictions may be imposed.

Arabic target:
وحُظرت الرقابة التي كانت مفروضة سابقاً، وينص كل من الدستور والقوانين ذات الصلة صراحة على الظروف التي تُفرض فيها قيود.

English source:
According to the contract ' s schedules, approximately 92 per cent of the contract value related to the supply of goods, and approximately 8 per cent related to services.

Arabic target:
وبحسب البرامج الزمنية المنصوص عليها في العقد، فإن نسبة نحو 92 في المائة من قيمة العقد تتصل بإمداد المواد وما يقرب من 8 في المائة بالخدمات.

English source:
persons who reached pensionable age or were recognized as disabled while raising children of the deceased person who were receiving or were entitled to receive orphan ' s (survivor ' s) pension.

Arabic target:
- الأشخاص الذين بلغوا سن المعاش أو أصبحوا عاجزين أثناء قيامهم بتربية أطفال الشخص المتوفى الذين يحصلون أو يحق لهم أن يتقاضوا معاش ا

In [17]:
train_df, temp_df = train_test_split(
    df,
    test_size=0.20,
    random_state=42
)

validation_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    random_state=42
)

train_df = train_df.reset_index(drop=True)
validation_df = validation_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

In [18]:
print("Training examples:", len(train_df))
print("Validation examples:", len(validation_df))
print("Test examples:", len(test_df))

print(
    "\nTotal examples:",
    len(train_df)
    + len(validation_df)
    + len(test_df)
)

Training examples: 32000
Validation examples: 4000
Test examples: 4000

Total examples: 40000


In [19]:
# Verify that the data split is correct
assert len(train_df) + len(validation_df) + len(test_df) == len(df)

# Verify that no missing values exist
for split_name, split_df in [
    ("Train", train_df),
    ("Validation", validation_df),
    ("Test", test_df)
]:
    assert split_df["english_text"].isnull().sum() == 0
    assert split_df["arabic_text"].isnull().sum() == 0
    assert (split_df["english_text"] == "").sum() == 0
    assert (split_df["arabic_text"] == "").sum() == 0

print("All data split checks passed successfully.")

All data split checks passed successfully.


In [20]:
train_dataset = Dataset.from_pandas(
    train_df,
    preserve_index=False
)

validation_dataset = Dataset.from_pandas(
    validation_df,
    preserve_index=False
)

test_dataset = Dataset.from_pandas(
    test_df,
    preserve_index=False
)

prepared_dataset = DatasetDict({
    "train": train_dataset,
    "validation": validation_dataset,
    "test": test_dataset
})

print(prepared_dataset)

DatasetDict({
    train: Dataset({
        features: ['arabic_text', 'english_text'],
        num_rows: 32000
    })
    validation: Dataset({
        features: ['arabic_text', 'english_text'],
        num_rows: 4000
    })
    test: Dataset({
        features: ['arabic_text', 'english_text'],
        num_rows: 4000
    })
})


In [21]:
print("Final columns:")
print(prepared_dataset["train"].column_names)

print("\nExample prepared training sample:")
print(prepared_dataset["train"][0])

Final columns:
['arabic_text', 'english_text']

Example prepared training sample:
{'arabic_text': 'وتحقيقاً لذلك، قام الممثل بزيارة مخيمات ومستوطنات المشردين داخلياً في الخرطوم وحولها في شيكان، والفتح 2 ومايو، واتجه إلى أبيي وكادوغلي وملكال وملوالكون ورومبك وجوبا.', 'english_text': 'To this end, the Representative visited Khartoum and the surrounding IDP camps and settlements at Shikan, Al Fatah 3 and Mayo, and travelled to Abyei, Kadugli, Malakal, Malualkon, Rumbek and Juba.'}


In [22]:
print("Translation direction verification:")

print("\nEnglish source:")
print(prepared_dataset["train"][0]["english_text"])

print("\nArabic target:")
print(prepared_dataset["train"][0]["arabic_text"])

Translation direction verification:

English source:
To this end, the Representative visited Khartoum and the surrounding IDP camps and settlements at Shikan, Al Fatah 3 and Mayo, and travelled to Abyei, Kadugli, Malakal, Malualkon, Rumbek and Juba.

Arabic target:
وتحقيقاً لذلك، قام الممثل بزيارة مخيمات ومستوطنات المشردين داخلياً في الخرطوم وحولها في شيكان، والفتح 2 ومايو، واتجه إلى أبيي وكادوغلي وملكال وملوالكون ورومبك وجوبا.


In [23]:
print("=" * 60)
print("PART 1: SETUP + DATA PREPARATION COMPLETED")
print("=" * 60)

print("\nDataset splits:")
print(
    f"Train: {len(train_df)} "
    f"({len(train_df) / len(df) * 100:.1f}%)"
)
print(
    f"Validation: {len(validation_df)} "
    f"({len(validation_df) / len(df) * 100:.1f}%)"
)
print(
    f"Test: {len(test_df)} "
    f"({len(test_df) / len(df) * 100:.1f}%)"
)

print("\nTotal samples:")
print(len(df))

print("\nFinal columns:")
print(prepared_dataset["train"].column_names)

print("\nTranslation direction:")
print("English source -> Arabic target")

print("\nPart 1 completed successfully.")

PART 1: SETUP + DATA PREPARATION COMPLETED

Dataset splits:
Train: 32000 (80.0%)
Validation: 4000 (10.0%)
Test: 4000 (10.0%)

Total samples:
40000

Final columns:
['arabic_text', 'english_text']

Translation direction:
English source -> Arabic target

Part 1 completed successfully.


==========================================================================

In [24]:
# part 2
# Load tokenizer and model
model_name = 'google/mt5-small'
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name, tie_word_embeddings=False)

print("Model loaded:", model_name)
print("Vocab size:", tokenizer.vocab_size)

Loading weights:   0%|          | 0/192 [00:00<?, ?it/s]

Model loaded: google/mt5-small
Vocab size: 250100


In [25]:
# inspect token lengths
print("===== ENGLISH =====")

en_lengths = [len(tokenizer.encode(t)) for t in df["english_text"]]
ar_lengths = [len(tokenizer.encode(t)) for t in df["arabic_text"]]

for p in [50, 75, 90, 95, 97.5, 99, 100]:
    print(f"{p:>5}%:", np.percentile(en_lengths, p))


print("\n===== ARABIC =====")

for p in [50, 75, 90, 95, 97.5, 99, 100]:
    print(f"{p:>5}%:", np.percentile(ar_lengths, p))


print("\n===== TRUNCATION RATES =====")

for L in [64, 96, 128, 160, 192, 256]:
    en_rate = np.mean(np.array(en_lengths) > L)
    ar_rate = np.mean(np.array(ar_lengths) > L)

    print(
        f"L={L:3} | "
        f"EN truncated={en_rate:.2%} | "
        f"AR truncated={ar_rate:.2%}"
    )

===== ENGLISH =====
   50%: 40.0
   75%: 57.0
   90%: 78.0
   95%: 95.0
 97.5%: 112.02500000000146
   99%: 141.0
  100%: 363.0

===== ARABIC =====
   50%: 55.0
   75%: 79.0
   90%: 107.0
   95%: 129.0
 97.5%: 154.0
   99%: 189.0
  100%: 476.0

===== TRUNCATION RATES =====
L= 64 | EN truncated=17.99% | AR truncated=38.52%
L= 96 | EN truncated=4.71% | AR truncated=13.79%
L=128 | EN truncated=1.45% | AR truncated=5.13%
L=160 | EN truncated=0.52% | AR truncated=2.09%
L=192 | EN truncated=0.18% | AR truncated=0.91%
L=256 | EN truncated=0.03% | AR truncated=0.22%


English (source): truncation basically collapses after 128 (1.45% → 0.52% at 160 is a small further gain for 25% more compute). 128 is the clear elbow point.

Arabic (target): truncation is still meaningfully worse at every length compared to EN — 5.13% at 128 is arguably too high for a target sequence, since a truncated label directly damages what the model is trained to predict (unlike a truncated source, which just loses some context). 160 brings it down to 2.09%, a solid improvement for a manageable size bump.

In [26]:
MAX_SOURCE_LEN = 128
MAX_TARGET_LEN = 160

In [27]:
def preprocess_function(examples):
    model_inputs = tokenizer(
        examples["english_text"],
        max_length=MAX_SOURCE_LEN,
        truncation=True,
    )
    labels = tokenizer(
        text_target=examples["arabic_text"],
        max_length=MAX_TARGET_LEN,
        truncation=True,
    )
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

tokenized_dataset = prepared_dataset.map(
    preprocess_function,
    batched=True,
    remove_columns=prepared_dataset["train"].column_names,
)

print(tokenized_dataset)
print(tokenized_dataset["train"][0])

Map:   0%|          | 0/32000 [00:00<?, ? examples/s]

Map:   0%|          | 0/4000 [00:00<?, ? examples/s]

Map:   0%|          | 0/4000 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 32000
    })
    validation: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 4000
    })
    test: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 4000
    })
})
{'input_ids': [926, 714, 3162, 261, 287, 259, 111240, 259, 65918, 121163, 476, 606, 305, 287, 259, 42296, 347, 259, 227535, 13861, 263, 305, 259, 263, 94768, 263, 344, 8243, 502, 261, 1151, 77804, 334, 381, 305, 42177, 261, 305, 11872, 11665, 288, 2922, 1347, 266, 261, 18412, 273, 3191, 261, 17509, 5360, 261, 154654, 473, 3160, 261, 19292, 17198, 305, 122976, 262, 260, 1], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], 'labels': [6427, 15617, 632, 1093, 259, 572, 3971, 343, 2

English sentence
      =>
   tokenizer
      =>
  input_ids
      =>
    mT5

Arabic sentence
      =>
   tokenizer
      =>
    labels
      =>
"correct answer"

In [28]:
'''We add padding because the sequences in a batch can have different lengths,
   but the model needs them to have the same shape. For the padded positions in
   the labels, we use -100, which tells the loss function to ignore those
   positions during training'''

data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
    padding=True,
    label_pad_token_id=-100,
)

print('data collator defined and part 2 is now done')


data collator defined and part 2 is now done


# Part 3: Training & Validation

Fine-tune mT5 using the data and model prepared in Parts 1 and 2.

Use the validation split to monitor loss. Keep the test split for Part 4.

## 3.1 Check Setup

Import training libraries and check GPU availability, required objects, and dataset columns.

In [1]:
import transformers
from transformers import Seq2SeqTrainingArguments, Seq2SeqTrainer, set_seed

print("Transformers:", transformers.__version__)
print("Imports OK")

Transformers: 5.17.0
Imports OK


In [29]:
import math
from pathlib import Path

import torch
import pandas as pd

required = [
    "model",
    "tokenizer",
    "tokenized_dataset",
    "data_collator",
    "MAX_TARGET_LEN",
]

missing = [name for name in required if name not in globals()]

if missing:
    raise RuntimeError(f"Run Parts 1 and 2 first. Missing: {missing}")

if not torch.cuda.is_available():
    raise RuntimeError(
        "Select a GPU runtime in Colab, then run the notebook again."
    )

for split in ["train", "validation"]:
    assert len(tokenized_dataset[split]) > 0, f"Empty split: {split}"

    assert {"input_ids", "attention_mask", "labels"}.issubset(
        tokenized_dataset[split].column_names
    ), f"Missing tokenized columns in {split}"

set_seed(42)

print("Transformers:", transformers.__version__)
print("GPU:", torch.cuda.get_device_name(0))
print("Training examples:", len(tokenized_dataset["train"]))
print("Validation examples:", len(tokenized_dataset["validation"]))

Transformers: 5.17.0
GPU: Tesla T4
Training examples: 32000
Validation examples: 4000


## 3.2 Training Settings

- Train for **1 epoch**.
- Learning rate: `5e-5`.
- Batch size: `2`, with `8` accumulation steps.
- Use FP32 and gradient checkpointing.
- Use Adafactor and approximately 10% warmup steps.
- Evaluate and save at the end of the epoch.

Files saved under `/content` are temporary.

In [30]:
OUTPUT_DIR = "/content/mt5_en_ar_training"
FINAL_MODEL_DIR = str(Path(OUTPUT_DIR) / "best_model")

model.config.use_cache = False

training_args = Seq2SeqTrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=1,
    learning_rate=5e-5,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=8,
    weight_decay=0.01,
    warmup_steps=math.ceil(0.1 * 3 * math.ceil(len(tokenized_dataset["train"]) / (2 * 8))),
    lr_scheduler_type="linear",
    optim="adafactor",
    max_grad_norm=1.0,
    fp16=False,
    bf16=False,
    gradient_checkpointing=True,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    predict_with_generate=True,
    generation_max_length=MAX_TARGET_LEN,
    generation_num_beams=1,
    logging_strategy="steps",
    logging_steps=50,
    logging_first_step=True,
    logging_nan_inf_filter=False,
    report_to="none",
    seed=42,
    data_seed=42,
)

print("Effective batch size on one GPU:",
      training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps)


Effective batch size on one GPU: 16


## 3.3 Create the Trainer

Connect the model, tokenizer, datasets, data collator, and training settings using `Seq2SeqTrainer`.

In [31]:
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["validation"],
    processing_class=tokenizer,
    data_collator=data_collator,
)


## 3.4 Train

Run training for one epoch and save a checkpoint.

Leave `RESUME_FROM_CHECKPOINT = None` for the first run.

In [32]:
RESUME_FROM_CHECKPOINT = None  # Example: "/content/mt5_en_ar_training/checkpoint-1000"

train_result = trainer.train(resume_from_checkpoint=RESUME_FROM_CHECKPOINT)
trainer.log_metrics("train", train_result.metrics)
trainer.save_metrics("train", train_result.metrics)
trainer.save_state()

print("Best checkpoint:", trainer.state.best_model_checkpoint)
print("Best validation loss during training:", trainer.state.best_metric)


[transformers] `use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.


Epoch,Training Loss,Validation Loss


Epoch,Training Loss,Validation Loss
1,32.185291,2.943861


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

***** train metrics *****
  epoch                    =        1.0
  total_flos               =  1796648GF
  train_loss               =    60.1278
  train_runtime            = 0:57:44.05
  train_samples_per_second =      9.238
  train_steps_per_second   =      0.577
Best checkpoint: /content/mt5_en_ar_training/checkpoint-2000
Best validation loss during training: 2.943861246109009


## 3.5 Validate

Calculate validation loss after training.

Translation quality will be evaluated using BLEU and chrF in Part 4.

In [33]:
validation_metrics = trainer.evaluate(metric_key_prefix="eval")
if not math.isfinite(validation_metrics["eval_loss"]):
    raise RuntimeError("Validation loss is not finite. Check training logs before using the model.")

trainer.log_metrics("eval", validation_metrics)
trainer.save_metrics("eval", validation_metrics)


Training Loss,Validation Loss,Epoch
32.185291,2.943861,1


***** eval metrics *****
  eval_loss = 2.9439


## 3.6 Training History

Display training and validation losses, and save the logs as a CSV file.

In [34]:
history = pd.DataFrame(trainer.state.log_history)
train_history = history.loc[history["loss"].notna(), ["step", "epoch", "loss"]]
validation_history = history.loc[
    history["eval_loss"].notna(), ["step", "epoch", "eval_loss"]
]
print("Training loss history:")
display(train_history.reset_index(drop=True))
print("Validation loss history:")
display(validation_history.reset_index(drop=True))
history.to_csv(Path(OUTPUT_DIR) / "training_history.csv", index=False)


Training loss history:


,step,epoch,loss
0,1,0.0005,256.505188
1,50,0.0250,225.634566
2,100,0.0500,216.246738
3,150,0.0750,197.198652
4,200,0.1000,168.318164
5,250,0.1250,139.566094
6,300,0.1500,119.668506
7,350,0.1750,98.923682
8,400,0.2000,79.065693
9,450,0.2250,60.482144


Validation loss history:


,step,epoch,eval_loss
0,2000,1.0,2.943861
1,2000,1.0,2.943861


## 3.7 Save the Model

Save the trained model and tokenizer for Part 4.

Use a complete checkpoint to resume training.

In [35]:
model = trainer.model
model.gradient_checkpointing_disable()
model.config.use_cache = True
model.eval()

trainer.save_model(FINAL_MODEL_DIR)
tokenizer.save_pretrained(FINAL_MODEL_DIR)

print("Best model and tokenizer saved to:", FINAL_MODEL_DIR)
print("Part 3 completed. The test split is reserved for Part 4.")


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Best model and tokenizer saved to: /content/mt5_en_ar_training/best_model
Part 3 completed. The test split is reserved for Part 4.


In [36]:
from google.colab import drive
import shutil

drive.mount("/content/drive")

shutil.copytree(
    "/content/mt5_en_ar_training",
    "/content/drive/MyDrive/mini_project_2_backup/mt5_en_ar_training",
    dirs_exist_ok=True
)

print("Backup saved successfully!")

Mounted at /content/drive
Backup saved successfully!
